# 03 - Feature engineering
Builds one row per (GPS observation, candidate arc). Features use GPS, timing and network geometry only.
The scaler is fitted on **training** rows only and applied unchanged to student-testing.

In [ ]:
import sys, pathlib, warnings
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt

OUT, GRAPHS, MAPS = ROOT / "outputs", ROOT / "outputs" / "graphs", ROOT / "outputs" / "maps"
PROC = ROOT / "dataset" / "processed"
for d in (GRAPHS, MAPS, PROC, ROOT / "models" / "configurations"):
    d.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 200, "display.max_columns", 40)

In [ ]:
from src.feature_engineering import FEATURE_COLUMNS, FeatureConfig, build_dataset, save_dataset
import joblib
CFG = FeatureConfig()
N_TRAIN_TRIPS = None          # set e.g. 25 for a quick dry run
from src import data_loader as dl
train_ids = dl.list_trips("training")[:N_TRAIN_TRIPS]
tr_c, tr_o = build_dataset("training", train_ids, CFG, n_jobs=2)
st_c, st_o = build_dataset("student_testing", None, CFG, n_jobs=2)
save_dataset(PROC / "training_features.joblib", tr_c, tr_o, CFG)
save_dataset(PROC / "student_testing_features.joblib", st_c, st_o, CFG)
print(CFG)

## 1. Feature list and what each captures

In [ ]:
FEATURE_NOTES = {
 "dist_m": "GPS-to-arc distance (m)", "dist_rank": "rank of the arc by distance within the observation",
 "dist_gap_to_best": "distance minus nearest candidate distance", "dist_ratio_to_best": "distance / (nearest distance + 1)",
 "proj_frac": "position of the closest point along the arc (0..1)", "dist_to_arc_end_m": "metres left before the arc ends",
 "arc_len_m": "arc length", "n_cands": "candidates available for this observation",
 "heading_diff_deg": "|vehicle heading - arc bearing| (0..180)", "heading_cos": "cosine of that difference",
 "speed_mps": "windowed speed", "heading_reliable": "1 if the vehicle moved enough for a trustworthy heading",
 "step_prev_m": "distance from previous GPS fix", "dt_prev": "time since previous fix",
 "to_node_out_deg": "out-degree of the arc's end node", "from_node_in_deg": "in-degree of the arc's start node",
 "has_reverse_arc": "road also exists in the opposite direction",
 "neigh_presence": "share of +-3 neighbouring observations that also have this arc as a candidate",
 "neigh_mean_dist": "mean distance to this arc over those neighbours"}
assert list(FEATURE_NOTES) == FEATURE_COLUMNS
pd.Series(FEATURE_NOTES, name="meaning").to_frame()

## 2. Dataset summary (training vs student-testing)

In [ ]:
def summary(c, o, name):
    return {"partition": name, "trips": c.group.nunique(), "observations": len(o), "candidate_rows": len(c),
            "rows_per_obs": len(c) / len(o), "obs_without_candidates": float((o.n_cands == 0).mean()),
            "candidate_miss_rate": float((o.ref_arc < 0).mean()), "positive_share_of_rows": float(c.y[c.has_ref].mean())}
pd.DataFrame([summary(tr_c, tr_o, "training"), summary(st_c, st_o, "student_testing")]).round(4)

In [ ]:
# per-trip data quality: GPS far from every road (no candidates) or reference route far from the GPS (ref_miss)
trip_quality = tr_o.groupby("group").agg(n_obs=("obs_idx", "size"), no_candidate=("n_cands", lambda s: (s == 0).mean()),
                                          ref_miss=("ref_arc", lambda s: (s < 0).mean()), max_dt_s=("dt_prev", "max"))
trip_quality.to_csv(OUT / "training_trip_quality.csv")
print("trips with > 5% observations lacking a reference arc in the candidates:", int((trip_quality.ref_miss > 0.05).sum()))
trip_quality.sort_values("ref_miss", ascending=False).head(8).round(3)

In [ ]:
print("NaN/inf in feature matrix:", int((~np.isfinite(tr_c[FEATURE_COLUMNS].to_numpy(dtype=float))).sum()))
feat_summary = tr_c[FEATURE_COLUMNS].describe().T; feat_summary.to_csv(OUT / "feature_summary.csv"); feat_summary.round(3)

## 3. Scaler (fitted on training rows with a reference label only)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(tr_c.loc[tr_c.has_ref, FEATURE_COLUMNS])
joblib.dump(scaler, ROOT / "models" / "scaler.pkl")
pd.DataFrame({"mean": scaler.mean_, "std": scaler.scale_}, index=FEATURE_COLUMNS).round(3)

## 4. Feature behaviour: reference candidate vs other candidates (partition: training)

In [ ]:
lab = tr_c[tr_c.has_ref].sample(200_000, random_state=0) if len(tr_c) > 200_000 else tr_c[tr_c.has_ref]
show = ["dist_m", "heading_diff_deg", "neigh_presence", "neigh_mean_dist", "dist_gap_to_best", "dist_to_arc_end_m"]
fig, ax = plt.subplots(2, 3, figsize=(14, 7))
for a, f in zip(ax.ravel(), show):
    hi = lab[f].quantile(0.995)
    bins = np.linspace(lab[f].min(), hi, 40)
    a.hist(lab.loc[lab.y == 0, f].clip(upper=hi), bins=bins, alpha=.6, density=True, label="other candidates")
    a.hist(lab.loc[lab.y == 1, f].clip(upper=hi), bins=bins, alpha=.6, density=True, label="reference arc")
    a.set_title(f)
ax[0, 0].legend(); fig.tight_layout(); fig.savefig(GRAPHS / "feature_distributions.png"); plt.show()

In [ ]:
corr = lab[FEATURE_COLUMNS + ["y"]].corr()
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=7); ax.grid(False)
fig.colorbar(im); ax.set_title("Feature correlation (training)"); fig.tight_layout()
fig.savefig(GRAPHS / "feature_correlation.png"); plt.show()
print(corr["y"].drop("y").sort_values(key=abs, ascending=False).head(8).round(3))

## 5. Reasoning (fill in)
* Which features separate the reference arc best and why:
* Features you tried and dropped:
* Leakage check (nothing uses the route / labels): 